**Dataset:** We're using the Diabetes dataset, which contains health measurements to predict diabetes progression.
**Model:** A Linear Regression model is used. It finds a straight-line relationship between the measurements and disease progression.
MLflow: We're using MLflow to track the experiment, logging the Mean Squared Error and saving the trained model.

In [19]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # New model: Linear Regression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --------------------------------------------------------------------
# 1. Set Experiment Name
mlflow.set_experiment("car_mpg_linear_regression")

# --------------------------------------------------------------------
# 2. Load Car Dataset (Auto MPG - predicting fuel efficiency)
print("Loading car dataset...")
auto_data = fetch_openml(name='autoMpg', version=1, as_frame=True, parser='auto')

# Prepare features and target
X = auto_data.data
y = auto_data.target

# Combine and drop missing values
df = X.copy()
df['target'] = y
df = df.dropna()

# Drop non-numeric identifier column (car_name was not in the DataFrame X)
X = df.drop(columns=['target'])
y = df['target']

# Convert categorical columns (e.g., origin) to numerical values
X = X.apply(lambda col: col.cat.codes if col.dtype.name == 'category' else col)

# Split dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling (Recommended for Linear Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Sample Size:", len(X_train))
print("Testing Sample Size:", len(X_test))

# --------------------------------------------------------------------
# 3. Train Model & Log Experiment with MLflow
with mlflow.start_run():

    # Initialize Linear Regression model
    model = LinearRegression()

    # Fit model on scaled data
    model.fit(X_train_scaled, y_train)

    # Predict and Evaluate
    predictions = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    # Log Parameters
    mlflow.log_param("algorithm", "LinearRegression")
    mlflow.log_param("fit_intercept", model.fit_intercept)

    # Log Metrics
    mlflow.log_metric("Mean Absolute Error", mae)
    mlflow.log_metric("Mean Squared Error", mse)
    mlflow.log_metric("R2 Score", r2)

    # Log Trained Model
    mlflow.sklearn.log_model(model, artifact_path="Linear_Regression_Car_Model")

    print("\n--- Model Evaluation ---")
    print("Mean Absolute Error (MAE):", round(mae, 4))
    print("Mean Squared Error (MSE):", round(mse, 4))
    print("R² Score:", round(r2, 4))

2026/07/30 09:04:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Loading car dataset...
Training Sample Size: 313
Testing Sample Size: 79

--- Model Evaluation ---
Mean Absolute Error (MAE): 2.3618
Mean Squared Error (MSE): 10.1732
R² Score: 0.8007
